In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/PSICHIC/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/PSICHIC'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/PSICHIC/script')



In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict

# Define all bands
BANDS = [
    "band_0.10_0.50",
    "band_0.20_0.60",
    "band_0.30_0.70",
    "band_0.40_0.80",
    "band_0.50_0.90",
    "band_0.60_1.00"
]

EPOCHS = 200
BASE_DATASET_PATH = _kp('dataset/data_split_from_merged_true_fold')
DATE_SUFFIX = datetime.now().strftime("%y%m%d")

# Main log path
MASTER_LOG_PATH = _kp('dataset/data_split_from_merged_true_fold')
MASTER_LOG_FILE = os.path.join(MASTER_LOG_PATH, f"all_bands_ef_results_{DATE_SUFFIX}.log")

def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50]):
    """
    Calculate the Enrichment Factor (EF).

    Args:
        y_true: Binary labels (0 or 1; 1 indicates an active compound).
        y_score: Prediction scores; higher values indicate greater predicted activity.
        percentages: Percentages at which EF is calculated.

    Returns:
        A dictionary mapping each percentage to its EF.
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    
    # Validate the inputs
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have equal lengths")
    
    N = len(y_true)  # Total number of compounds
    n_active = np.sum(y_true == 1)  # Total number of active compounds
    
    if n_active == 0:
        print("Warning: no active compounds; returning EF=0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    # Sort by descending prediction score
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        # Calculate the number of compounds in the top x%
        n_selected = max(1, int(N * percentage / 100))  # Select at least one compound
        
        # Count active compounds in the selected subset
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        
        # Calculate the enrichment factor
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        
        results[f"EF_{percentage}%"] = ef_value
    
    return results

def calculate_ef_from_prediction_file(prediction_file, master_log):
    """Calculate EF from a prediction file."""
    try:
        if not os.path.exists(prediction_file):
            msg = f"  Warning: Prediction file does not exist - {prediction_file}\n"
            print(msg, end='')
            master_log.write(msg)
            return None
        
        df = pd.read_csv(prediction_file)
        
        msg = f"  Prediction file: {os.path.basename(prediction_file)}\n"
        msg += f"  Total samples: {len(df)}\n"
        msg += f"  Active compounds: {np.sum(df['classification_label'] == 1)}\n"
        msg += f"  Inactive compounds: {np.sum(df['classification_label'] == 0)}\n"
        print(msg, end='')
        master_log.write(msg)
        
        # Calculate EF
        ef_results = calculate_ef(
            y_true=df['classification_label'].values,
            y_score=df['predicted_binary_interaction'].values
        )
        
        return ef_results
        
    except Exception as e:
        msg = f"  Error computing EF: {str(e)}\n"
        print(msg, end='')
        master_log.write(msg)
        return None

def run_single_fold(band, fold, master_log):
    """Train a single fold and wait for completion."""
    datafolder = f"{BASE_DATASET_PATH}/{band}/train_valid_split/train_valid_{fold}"
    result_path = _kp(f"model/PSICHIC/{band}/train_valid_{fold}")
    log_file = f"{result_path}/PSICHIC_SPECTRA_{band}_fold_{fold}_{DATE_SUFFIX}.log"
    
    # Create the results directory
    os.makedirs(result_path, exist_ok=True)
    
    msg = f"\n{'='*80}\n"
    msg += f"Starting run: {band} - Fold {fold}\n"
    msg += f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    msg += f"Data directory: {datafolder}\n"
    msg += f"Results path: {result_path}\n"
    msg += f"Training log: {log_file}\n"
    msg += f"{'='*80}\n"
    
    print(msg, end='')
    master_log.write(msg)
    master_log.flush()
    
    # Build the command
    cmd = [
        _ks_sys.executable, _kp('external/PSICHIC/main.py'),
        "--datafolder", datafolder,
        "--epochs", str(EPOCHS),
        "--result_path", result_path,
        "--classification_task", "True"
    ]
    
    # Open the log and run the command to completion
    with open(log_file, 'w') as log:
        process = subprocess.run(
            cmd,
            stdout=log,
            stderr=subprocess.STDOUT
        )
    
    status_msg = ""
    if process.returncode == 0:
        status_msg = f"✓ {band} - Fold {fold} Training complete\n"
    else:
        status_msg = f"✗ {band} - Fold {fold} Training failed (return code: {process.returncode})\n"
    
    print(status_msg, end='')
    master_log.write(status_msg)
    master_log.flush()
    
    # Calculate EF if training succeeded
    ef_results = None
    if process.returncode == 0:
        # Locate the prediction file
        prediction_file = os.path.join(result_path, "test_prediction_seed1.csv")
        
        # Try alternative prediction files if the default is absent
        if not os.path.exists(prediction_file):
            import glob
            prediction_files = glob.glob(os.path.join(result_path, "*prediction*.csv"))
            if prediction_files:
                prediction_file = prediction_files[0]
        
        msg = f"\nComputing EF: {band} - Fold {fold}\n"
        msg += f"{'-'*80}\n"
        print(msg, end='')
        master_log.write(msg)
        master_log.flush()
        
        ef_results = calculate_ef_from_prediction_file(prediction_file, master_log)
        
        if ef_results:
            msg = "  EF results:\n"
            for key, value in sorted(ef_results.items()):
                msg += f"    {key}: {value:.4f}\n"
            msg += "\n"
            print(msg, end='')
            master_log.write(msg)
            master_log.flush()
    
    return process.returncode, ef_results

def calculate_statistics(ef_data):
    """Calculate EF means and standard deviations."""
    stats = {}
    
    if not ef_data:
        return stats
    
    # Collect all EF metrics
    ef_keys = list(ef_data[0].keys())
    
    for key in ef_keys:
        values = [ef[key] for ef in ef_data if ef is not None]
        if values:
            stats[key] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'values': values
            }
    
    return stats

def main():
    """Run all tasks sequentially."""
    
    # Create the main log file
    os.makedirs(MASTER_LOG_PATH, exist_ok=True)
    
    # Store all EF results
    all_ef_results = defaultdict(list)  # {band: [ef_dict1, ef_dict2, ...]}
    
    with open(MASTER_LOG_FILE, 'w', buffering=1) as master_log:
        start_time = datetime.now()
        
        header = f"{'='*80}\n"
        header += f"PSICHIC training and EF evaluation across similarity bands with 5-fold cross-validation\n"
        header += f"{'='*80}\n"
        header += f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        header += f"Log file: {MASTER_LOG_FILE}\n"
        header += f"Training epochs: {EPOCHS}\n"
        header += f"Number of similarity bands: {len(BANDS)}\n"
        header += f"Folds per similarity band: 5\n"
        header += f"Total tasks: {len(BANDS) * 5}\n"
        header += f"{'='*80}\n\n"
        
        print(header, end='')
        master_log.write(header)
        
        results = []
        
        # Iterate over bands and folds
        for band_idx, band in enumerate(BANDS, 1):
            band_header = f"\n{'#'*80}\n"
            band_header += f"Similarity band {band_idx}/{len(BANDS)}: {band}\n"
            band_header += f"{'#'*80}\n"
            print(band_header, end='')
            master_log.write(band_header)
            master_log.flush()
            
            for fold in range(1, 6):
                return_code, ef_results = run_single_fold(band, fold, master_log)
                results.append({
                    'band': band,
                    'fold': fold,
                    'status': 'success' if return_code == 0 else 'failed',
                    'return_code': return_code
                })
                
                # Save EF results
                if ef_results is not None:
                    all_ef_results[band].append(ef_results)
        
        # Print the aggregate results
        end_time = datetime.now()
        duration = end_time - start_time
        
        summary = f"\n\n{'='*80}\n"
        summary += f"All tasks completed\n"
        summary += f"{'='*80}\n"
        summary += f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        summary += f"Total duration: {duration}\n"
        summary += f"{'='*80}\n\n"
        
        # Aggregate statistics
        success_count = sum(1 for r in results if r['status'] == 'success')
        failed_count = len(results) - success_count
        
        summary += f"Training results summary:\n"
        summary += f"  Succeeded: {success_count}/{len(results)}\n"
        summary += f"  Failed: {failed_count}/{len(results)}\n"
        
        if failed_count > 0:
            summary += f"\nFailed tasks:\n"
            for r in results:
                if r['status'] == 'failed':
                    summary += f"  - {r['band']} Fold {r['fold']} (Return code: {r['return_code']})\n"
        
        summary += f"\n"
        print(summary, end='')
        master_log.write(summary)
        
        # Calculate and print EF statistics for each band
        ef_summary = f"\n{'='*80}\n"
        ef_summary += f"EF summary (5-fold cross-validation for each similarity band)\n"
        ef_summary += f"{'='*80}\n\n"
        
        print(ef_summary, end='')
        master_log.write(ef_summary)
        
        for band in BANDS:
            if band in all_ef_results and all_ef_results[band]:
                stats = calculate_statistics(all_ef_results[band])
                
                band_summary = f"{band}:\n"
                band_summary += f"{'-'*80}\n"
                
                if stats:
                    for key in sorted(stats.keys()):
                        mean = stats[key]['mean']
                        std = stats[key]['std']
                        values = stats[key]['values']
                        band_summary += f"  {key}:\n"
                        band_summary += f"    Mean ± standard deviation: {mean:.4f} ± {std:.4f}\n"
                        band_summary += f"    Per fold: {', '.join([f'{v:.4f}' for v in values])}\n"
                else:
                    band_summary += "  No valid EF data\n"
                
                band_summary += "\n"
                print(band_summary, end='')
                master_log.write(band_summary)
            else:
                msg = f"{band}:\n  No EF data\n\n"
                print(msg, end='')
                master_log.write(msg)
        
        # Save EF results to CSV
        ef_csv_file = os.path.join(MASTER_LOG_PATH, f"ef_summary_{DATE_SUFFIX}.csv")
        try:
            summary_data = []
            for band in BANDS:
                if band in all_ef_results and all_ef_results[band]:
                    stats = calculate_statistics(all_ef_results[band])
                    row = {'Band': band}
                    for key in sorted(stats.keys()):
                        row[f'{key}_mean'] = stats[key]['mean']
                        row[f'{key}_std'] = stats[key]['std']
                    summary_data.append(row)
            
            if summary_data:
                df_summary = pd.DataFrame(summary_data)
                df_summary.to_csv(ef_csv_file, index=False)
                msg = f"{'='*80}\n"
                msg += f"EF statistics CSV saved: {ef_csv_file}\n"
                msg += f"{'='*80}\n"
                print(msg, end='')
                master_log.write(msg)
        except Exception as e:
            msg = f"\nError saving EF statistics CSV: {str(e)}\n"
            print(msg, end='')
            master_log.write(msg)

if __name__ == "__main__":
    import sys
    # Flush output when running under nohup
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
    
    main()